# PySQReg: Spatial Quantile Regression in Python

**A complete tutorial** — from theory to estimation, diagnostics, and visualisation.

This notebook walks through the full workflow of the `pysqreg` library:

1. Mathematical foundations of the SAR quantile model
2. Simulating spatial data on a regular lattice
3. Testing for spatial autocorrelation (Moran's I)
4. Estimating the model with two IV strategies
5. Exploring how effects vary across quantiles (quantile process)
6. Interpreting spatial impacts (direct and spillover effects)
7. Residual diagnostics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix, csc_matrix, diags
from scipy.sparse import eye as speye
from scipy.sparse.linalg import splu
from scipy.stats import chi2

from pysqreg import QuantSAR, moran_test
from pysqreg.plots import (
    plot_moran,
    fit_quantile_process,
    plot_quantile_process,
    plot_rho_path,
)

np.random.seed(42)

---
## 1. The Spatial Autoregressive Quantile Model

### Standard quantile regression

Classical **quantile regression** (Koenker & Bassett, 1978) models the $\tau$-th conditional quantile of $y$ given covariates $X$:

$$
Q_\tau(y_i \mid \mathbf{x}_i) = \mathbf{x}_i'\,\boldsymbol{\beta}(\tau), \qquad \tau \in (0,1)
$$

Unlike OLS, which targets the conditional **mean**, quantile regression traces the **entire conditional distribution** — revealing whether covariates affect the lower tail, the median, or the upper tail differently.

### Adding spatial dependence

When observations are arranged on a **lattice** (grid, administrative regions, etc.), neighbours' outcomes influence one another.  The **Spatial Autoregressive (SAR)** quantile model captures this:

$$
\boxed{\;\mathbf{y} = \rho\, W\mathbf{y} + X\boldsymbol{\beta} + \mathbf{u},
\qquad Q_\tau(\mathbf{u} \mid X) = \mathbf{0}\;}
$$

| Symbol | Meaning |
|--------|---------|
| $\mathbf{y}$ | $(n \times 1)$ dependent variable |
| $W$ | $(n \times n)$ row-standardised spatial weight matrix |
| $\rho$ | spatial autoregressive parameter ($\lvert\rho\rvert < 1$) |
| $X$ | $(n \times k)$ matrix of explanatory variables |
| $\boldsymbol{\beta}$ | $(k \times 1)$ coefficient vector |
| $\mathbf{u}$ | disturbance whose $\tau$-quantile is zero conditional on $X$ |

### The reduced form

Solving for $\mathbf{y}$:

$$
\mathbf{y} = \underbrace{(I_n - \rho W)^{-1}}_{S^{-1}}\, X\boldsymbol{\beta}
           \;+\; S^{-1}\,\mathbf{u}
$$

The **multiplier matrix** $S^{-1}$ propagates both the systematic component and the shocks across space — a change in region $j$ ripples through $W$ to affect region $i$.

### The endogeneity problem

The spatial lag $W\mathbf{y}$ is **endogenous**: since $\mathbf{y}$ depends on $\mathbf{u}$, so does $W\mathbf{y}$.  Naively including $W\mathbf{y}$ as a regressor produces inconsistent estimates.

**Solution — Instrumental Variables.**  
Under standard regularity conditions, the columns of $[X,\; WX,\; W^2X,\; \ldots]$ are valid instruments for $W\mathbf{y}$: they are correlated with the spatial lag (relevance) but uncorrelated with $\mathbf{u}$ (exogeneity).  PySQReg uses $Z = [X,\; WX]$ by default.

### Two estimation strategies

PySQReg implements two IV-based approaches:

---

**Method 1 — Kim & Muller (2004): Two-Stage IV Quantile Regression**

| Stage | Procedure |
|-------|-----------|
| 1 | Quantile-regress $W\mathbf{y}$ on instruments $Z$ to obtain $\widehat{W\mathbf{y}}$ |
| 2 | Quantile-regress $\mathbf{y}$ on $[X,\; \widehat{W\mathbf{y}}]$ to obtain $(\hat{\boldsymbol{\beta}},\hat{\rho})$ |

Inference is via the **bootstrap**: resample $(y_i, \mathbf{x}_i, w_{i\cdot})$ with replacement $B$ times and compute standard errors from the empirical distribution of the estimates.

---

**Method 2 — Chernozhukov & Hansen (2006): Grid-Search IVQR**

For each candidate $\rho$ in a fine grid:

1. Transform the dependent variable: $\tilde{y}(\rho) = y - \rho\,Wy$
2. Quantile-regress $\tilde{y}(\rho)$ on $[X, \widehat{Wy}]$, storing the coefficient $\alpha(\rho)$ on $\widehat{Wy}$
3. Select $\hat{\rho} = \arg\min_{\rho}|\alpha(\rho)|$

The optimal $\rho$ is the value that makes the instrument **irrelevant** in the transformed regression — the signature of correct endogeneity removal.  Inference uses a **sandwich variance estimator** with the Bofinger (1975) bandwidth for sparsity estimation.

---
## 2. Simulating Spatial Data

We generate data from a known DGP so we can verify the estimator recovers the truth.

**Design choices:**
- **Grid:** $15 \times 15 = 225$ locations (queen contiguity)
- **Heteroscedastic, skewed errors:** $u_i = (1 + 0.3\,x_{1i})\,(\varepsilon_i - 3)$, $\;\varepsilon_i \sim \chi^2(3)$  
  This guarantees that quantile regression gives **genuinely different** results from OLS.
- **True parameters:** $\alpha = 5$, $\beta_1 = 2$, $\beta_2 = -1$, $\rho = 0.4$

Because the error variance grows with $x_1$, the **true quantile-varying slopes** are:

$$
\beta_1(\tau) = 2 + 0.3\,\bigl[F_{\chi^2(3)}^{-1}(\tau) - 3\bigr]
\qquad\text{and}\qquad
\beta_2(\tau) = -1 \;\;\text{(constant)}
$$

So $\beta_1$ increases with $\tau$ (larger effect in the upper tail), while $\beta_2$ is stable.

In [ ]:
def queen_weights(nrow, ncol):
    """Row-standardised queen contiguity weight matrix for a regular grid."""
    n = nrow * ncol
    W = lil_matrix((n, n))
    for i in range(nrow):
        for j in range(ncol):
            idx = i * ncol + j
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0:
                        continue
                    ni, nj = i + di, j + dj
                    if 0 <= ni < nrow and 0 <= nj < ncol:
                        W[idx, ni * ncol + nj] = 1.0
    W = W.tocsr()
    row_sums = np.array(W.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0
    return (diags(1.0 / row_sums) @ W).tocsc()

In [ ]:
# ── Grid & weight matrix ─────────────────────────────────────────────
nrow, ncol = 15, 15
n = nrow * ncol
W = queen_weights(nrow, ncol)

# ── Covariates ────────────────────────────────────────────────────────
rng = np.random.default_rng(2024)
x1 = rng.uniform(0, 10, n)
x2 = rng.normal(0, 1, n)
X = np.column_stack([x1, x2])

# ── True parameters ───────────────────────────────────────────────────
rho_true   = 0.4
alpha_true = 5.0
beta_true  = np.array([2.0, -1.0])

# ── Heteroscedastic, skewed errors ────────────────────────────────────
sigma   = 1 + 0.3 * x1
epsilon = sigma * (rng.chisquare(3, n) - 3)

# ── Generate y from the reduced form ──────────────────────────────────
A  = speye(n, format='csc') - rho_true * W
nu = alpha_true + X @ beta_true + epsilon
y  = splu(A).solve(nu)

print(f"n = {n}")
print(f"y  : mean={y.mean():.2f}, std={y.std():.2f}")
print(f"x1 : mean={x1.mean():.2f}, std={x1.std():.2f}")
print(f"x2 : mean={x2.mean():.2f}, std={x2.std():.2f}")

In [ ]:
# ── Visualise the spatial data on the lattice ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, vals, label, cmap in zip(
    axes,
    [y, x1, x2],
    [r'$y$ (dependent variable)', r'$x_1$', r'$x_2$'],
    ['YlOrRd', 'Blues', 'PuOr'],
):
    im = ax.imshow(vals.reshape(nrow, ncol), cmap=cmap, aspect='equal')
    ax.set_title(label, fontsize=12, fontweight='semibold', pad=8)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    r'Spatial Data on the $15 \times 15$ Lattice',
    fontsize=14, fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

---
## 3. Testing for Spatial Autocorrelation — Moran's I

Before modelling spatial dependence we should **detect** it.  Moran's I (Cliff & Ord, 1981) is the spatial analogue of a correlation coefficient:

$$
I \;=\; \frac{n}{S_0}\,\frac{\mathbf{z}'\,W\,\mathbf{z}}{\mathbf{z}'\mathbf{z}},
\qquad \mathbf{z} = \mathbf{x} - \bar{x},
\quad S_0 = \sum_{i,j} w_{ij}
$$

| Value | Interpretation |
|-------|----------------|
| $I > E[I] = -1/(n-1)$ | Positive autocorrelation (similar values cluster) |
| $I < E[I]$ | Negative autocorrelation (checkerboard pattern) |
| $I \approx E[I]$ | No significant spatial pattern |

The **Moran scatterplot** visualises this: the slope of the regression line through $(z_i, \, (Wz)_i)$ equals $I$ (for row-standardised $W$).  Points in the **HH** and **LL** quadrants contribute to positive autocorrelation (clusters), while **HL** and **LH** are spatial outliers.

In [ ]:
# ── Moran's I test on y ──────────────────────────────────────────────
result = moran_test(y, W)
result.summary()

In [ ]:
# ── Moran scatterplot ────────────────────────────────────────────────
plot_moran(y, W)
plt.show()

The significant positive Moran's I confirms strong spatial clustering in $y$ — justifying the SAR specification.

---
## 4. Model Estimation

### 4.1 Kim & Muller Two-Stage (bootstrap inference)

In [ ]:
model_ts = QuantSAR(tau=0.5, method='two_stage', nboot=300, random_state=42)
model_ts.fit(X, y, W)
model_ts.summary()

In [ ]:
print(f"\nTrue parameters:  alpha={alpha_true}, beta1={beta_true[0]}, beta2={beta_true[1]}, rho={rho_true}")
print(f"Estimated:        alpha={model_ts.intercept_:.3f}, "
      f"beta1={model_ts.coef_[0]:.3f}, beta2={model_ts.coef_[1]:.3f}, "
      f"rho={model_ts.rho_:.3f}")

### 4.2 Chernozhukov & Hansen Grid-Search (sandwich inference)

In [ ]:
model_gs = QuantSAR(tau=0.5, method='grid_search',
                     rhomat=np.arange(-0.9, 0.91, 0.01))
model_gs.fit(X, y, W)
model_gs.summary()

### 4.3 Visualising the rho path

The Chernozhukov-Hansen procedure selects $\hat{\rho}$ as the value minimising $|\alpha(\rho)|$.  The **rho path** shows how this coefficient varies over the grid — the zero-crossing identifies the optimal $\rho$.

In [ ]:
plot_rho_path(model_gs)
plt.show()

---
## 5. The Quantile Process

A single quantile gives one slice of the conditional distribution.  The **quantile process** estimates the model at a grid of quantiles $\tau \in \{0.05, 0.10, \ldots, 0.95\}$ and plots how each coefficient evolves:

$$
\tau \;\mapsto\; \hat{\boldsymbol{\beta}}(\tau)
$$

A **flat** process means the covariate has a uniform (location-shift) effect across the distribution — OLS would suffice.  A **sloped** process reveals distributional heterogeneity that only quantile regression can capture.

The **dashed grey line** shows the non-spatial OLS estimate for reference.

**Recall** the true DGP slopes:

$$
\beta_1(\tau) = 2 + 0.3\,\bigl[F_{\chi^2(3)}^{-1}(\tau) - 3\bigr]
\quad\text{(increasing)},
\qquad
\beta_2(\tau) = -1
\quad\text{(flat)}
$$

In [ ]:
# ── Fit the quantile process ──────────────────────────────────────────
taus = np.arange(0.10, 0.91, 0.05)

qp = fit_quantile_process(
    X, y, W,
    taus=taus,
    method='two_stage',
    nboot=200,
    random_state=42,
    verbose=1,
)

In [ ]:
# ── Plot the quantile process ─────────────────────────────────────────
fig = plot_quantile_process(
    qp,
    title='Quantile Process — Spatial QR (Kim & Muller)',
)

# Overlay the TRUE quantile-varying coefficients
true_beta1 = 2 + 0.3 * (chi2.ppf(taus, df=3) - 3)
true_beta2 = np.full_like(taus, -1.0)

axes = fig.axes
axes[0].plot(taus, true_beta1, color='#2CA02C', linewidth=1.5,
             linestyle=':', label='True $\\beta_1(\\tau)$', zorder=4)
axes[0].legend(fontsize=8.5, loc='upper left')

axes[1].plot(taus, true_beta2, color='#2CA02C', linewidth=1.5,
             linestyle=':', label='True $\\beta_2(\\tau)$', zorder=4)
axes[1].legend(fontsize=8.5, loc='upper left')

plt.show()

**Reading the plot:**

- **$x_1$** shows a clear upward trend — its effect on $y$ is roughly 1.3 at the 10th percentile but almost 3.0 at the 90th.  This heterogeneity would be invisible to OLS.
- **$x_2$** is approximately flat at $-1$, consistent with a pure location-shift effect.
- The **green dotted lines** are the true DGP values — the estimator tracks them closely.

---
## 6. Spatial Impacts — Direct and Spillover Effects

In a SAR model, a change in $x_{jk}$ (covariate $k$ in region $j$) affects **all** regions through the multiplier matrix:

$$
\frac{\partial \mathbf{y}}{\partial x_{jk}} = S^{-1}\,\beta_k,
\qquad S = I_n - \rho W
$$

LeSage & Pace (2009) decompose this into:

| Impact | Formula | Interpretation |
|--------|---------|----------------|
| **Direct** | $\frac{\beta_k}{n}\,\mathrm{tr}(S^{-1})$ | Average effect on own region |
| **Total** | $\frac{\beta_k}{n}\,\mathbf{1}'S^{-1}\mathbf{1}$ | Average effect including all feedback |
| **Indirect** | Total $-$ Direct | Average spillover to neighbours |

The **indirect effect** (spillover) is the policy-relevant quantity: it measures how much a local intervention propagates to surrounding regions through the spatial network.

In [ ]:
# ── Spatial impacts from the two-stage model ──────────────────────────
print("=" * 65)
print("  Spatial Impacts (LeSage & Pace) — Two-Stage at tau = 0.5")
print("=" * 65)
print(model_ts.impacts_.to_string(float_format=lambda x: f"{x:.5f}"))
print("=" * 65)

In [ ]:
# ── Spatial impacts from the grid-search model ────────────────────────
print("=" * 65)
print("  Spatial Impacts (LeSage & Pace) — Grid Search at tau = 0.5")
print("=" * 65)
print(model_gs.impacts_.to_string(float_format=lambda x: f"{x:.5f}"))
print("=" * 65)

---
## 7. Residual Diagnostics

If the model has correctly captured spatial dependence, the **residuals should be spatially uncorrelated**.  We test this with Moran's I on the residuals.

A significant Moran's I on residuals suggests remaining spatial structure — the model may need a richer specification (e.g., spatial error term, higher-order lags).

In [ ]:
# ── Compute residuals ──────────────────────────────────────────────────
y_hat_ts = model_ts.predict(X, W, y)
resid_ts = y - y_hat_ts

y_hat_gs = model_gs.predict(X, W, y)
resid_gs = y - y_hat_gs

print("Two-Stage residuals:")
moran_test(resid_ts, W).summary()
print()
print("Grid-Search residuals:")
moran_test(resid_gs, W).summary()

In [ ]:
# ── Moran scatterplots of residuals ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plot_moran(resid_ts, W, ax=axes[0],
           title='Residuals — Two-Stage')
plot_moran(resid_gs, W, ax=axes[1],
           title='Residuals — Grid Search')

plt.tight_layout()
plt.show()

In [ ]:
# ── Spatial residual map ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, resid, label in zip(
    axes,
    [resid_ts, resid_gs],
    ['Two-Stage residuals', 'Grid-Search residuals'],
):
    vmax = max(abs(resid.min()), abs(resid.max()))
    im = ax.imshow(resid.reshape(nrow, ncol), cmap='RdBu_r',
                   vmin=-vmax, vmax=vmax, aspect='equal')
    ax.set_title(label, fontsize=12, fontweight='semibold', pad=8)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

---
## Summary

| Step | Tool | What we learned |
|------|------|------------------|
| Moran's I | `moran_test`, `plot_moran` | Strong spatial clustering in $y$ |
| Median model | `QuantSAR(tau=0.5)` | Both methods recover $\rho \approx 0.4$, $\beta \approx$ truth |
| Rho path | `plot_rho_path` | Clean zero-crossing confirms identification |
| Quantile process | `fit_quantile_process` | $\beta_1$ increases with $\tau$; $\beta_2$ is flat |
| Spatial impacts | `model.impacts_` | Spillovers amplify the direct effect by ~60 % |
| Residual diagnostics | `moran_test(resid, W)` | No remaining spatial structure |

PySQReg provides the complete workflow for spatial quantile regression on lattice data — from detection through estimation, inference, and diagnostics — in a scikit-learn-compatible API with publication-quality visualisations.